In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


###**Read Stream from Bronze**

In [0]:
bronze_df = (
    spark.readStream
    .table("databricks_cata.bronze.patient_flow")
)

###**Define Schema**

In [0]:
schema = StructType([
    StructField("patient_id", StringType()),
    StructField("gender", StringType()),
    StructField("age", IntegerType()),
    StructField("department", StringType()),
    StructField("admission_time", StringType()),
    StructField("discharge_time", StringType()),
    StructField("bed_id", IntegerType()),
    StructField("hospital_id", IntegerType()),
])

###**Parse it to dataframe**

In [0]:

from pyspark.sql.functions import (
    col,
    from_json,
    to_timestamp,
    when,
    current_timestamp,
    floor,
    rand,
    lit
)

import builtins


# 1. Parse JSON data


parsed_df = (
    bronze_df
    .withColumn("data", from_json(col("raw_json"), schema))
    .select("data.*")
)



# 2. Convert columns to Timestamp


clean_df = (
    parsed_df
    .withColumn("admission_time", to_timestamp(col("admission_time")))
    .withColumn("discharge_time", to_timestamp(col("discharge_time")))
)



# 3. Handle invalid admission times
# Replace NULL or future admission times with current timestamp


clean_df = clean_df.withColumn(
    "admission_time",
    when(
        col("admission_time").isNull() |
        (col("admission_time") > current_timestamp()),
        current_timestamp()
    ).otherwise(col("admission_time"))
)



# 4. Handle invalid ages
# Replace ages greater than 100 with a random age between 1 and 90


clean_df = clean_df.withColumn(
    "age",
    when(
        col("age") > 100,
        floor(rand() * 90 + 1).cast("int")
    ).otherwise(col("age"))
)



# 5. Schema evolution
# Add missing expected columns


expected_cols = [
    "patient_id",
    "gender",
    "age",
    "department",
    "admission_time",
    "discharge_time",
    "bed_id",
    "hospital_id"
]

for col_name in expected_cols:
    if col_name not in clean_df.columns:
        clean_df = clean_df.withColumn(col_name, lit(None))



# 6. Configure Silver Unity Catalog table


silver_patient_flow = "databricks_cata.silver.silver_patient_flow"



# 7. Track processed records


processed_counts = []



# 8. Write each micro-batch to Silver


def write_silver_batch(batch_df, batch_id):

    # Count records in the current micro-batch
    batch_count = batch_df.count()

    if batch_count > 0:

        (
            batch_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(silver_patient_flow)
        )

    # Record the batch count
    processed_counts.append(batch_count)

    print(f"Batch {batch_id}: {batch_count} records processed")



# 9. Start Structured Streaming query


query = (
    clean_df.writeStream
    .foreachBatch(write_silver_batch)
    .option(
        "checkpointLocation",
        "abfss://silver@databrickssp.dfs.core.windows.net/_checkpoints/patient_flow_silver"
    )
    .trigger(availableNow=True)
    .start()
)



# 10. Wait for streaming execution to finish


query.awaitTermination()



# 11. Calculate total records processed


new_records_count = sum(processed_counts)

print("====================================")
print("Silver Layer Processing Completed")
print("New Silver Records:", new_records_count)
print("====================================")



# 12. Return record count to Azure Data Factory


dbutils.notebook.exit(str(new_records_count))



In [0]:
display(spark.read.table("databricks_cata.silver.silver_patient_flow"))